# 06 · Analysis: can we train on Solcast and forecast with Reuniwatt?
**Why:** Reuniwatt forecasts only exist from ~Jul 2026. Solcast has history from 2023 to 19 Nov 2025. If Reuniwatt's history doesn't arrive in time, could a model **trained on Solcast** be **run on Reuniwatt** forecasts?

**Look at the data before any model:**

| Part | Question |
|---|---|
| A · EDA | For every source (Solcast, Reuniwatt, `satellite_weather_sts`, plant sensors, target): which months have values, null %, distributions by month |
| B · Solcast mapping | Which Solcast table belongs to which plant |
| C · Same scale? | Each source vs the **plant sensors** (the common reference across all periods): bias, error, and distribution of the clear-sky index |
| D · Transfer test | Train on Solcast → predict with Reuniwatt (raw and calibrated), vs a Reuniwatt-trained model and Reuniwatt's own power forecast |

Naming: **pv1 = small plant (~935 MW)**, **pv2 = large plant (~1,584 MW)**. Target = actual MW. MAPE only where actual > 150 MW.
Results → `results/solcast_reuniwatt_transfer.txt` next to this notebook (**commit & push**). Writes one working table to `output_schema`.

In [ ]:
dbutils.widgets.text("output_schema", "ewec_dev_powerops.scratch_yash", "Output schema (pipeline tables)")
dbutils.widgets.text("satellite_table", "", "satellite_weather_sts full name (blank = search)")
dbutils.widgets.text("solcast_train_start", "2024-01-01", "Solcast training start")
dbutils.widgets.text("solcast_train_end", "2025-11-18", "Solcast training end")
dbutils.widgets.text("reuni_train_end", "2026-08-31", "Reuniwatt-model training end")
dbutils.widgets.text("test_start", "2026-09-01", "Test start (Reuniwatt period)")
dbutils.widgets.text("test_end", "", "Test end (blank = latest)")
dbutils.widgets.dropdown("daytime_mode", "clock", ["clock", "sun"], "Daytime: 06-18 Gulf or sun up")

In [ ]:
%pip install -q lightgbm

In [ ]:
import os, sys
REPO = os.path.abspath(os.path.join(os.getcwd(), "..", ".."))
sys.path.insert(0, os.path.join(REPO, "src"))
for m in [m for m in list(sys.modules) if m.startswith("uc2_solar")]:
    del sys.modules[m]
from datetime import datetime, date
from functools import reduce
import numpy as np
import pandas as pd
from pyspark.sql import DataFrame, Window
from pyspark.sql import functions as F
from uc2_solar.spark_utils import add_solar, ceil_to, gulf_to_utc
from uc2_solar.pipelines.data_engineering.nodes import _clean_mw, _detect_swap_date
from uc2_solar.metrics import score

spark.conf.set("spark.sql.session.timeZone", "UTC")
spark.conf.set("spark.sql.execution.arrow.pyspark.enabled", "true")
W = {k: dbutils.widgets.get(k).strip() for k in ["output_schema", "satellite_table", "solcast_train_start", "solcast_train_end",
                                                  "reuni_train_end", "test_start", "test_end", "daytime_mode"]}
S = W["output_schema"]
SITES = {"pv1": {"capacity_mw": 935, "lat": 24.45, "lon": 54.6}, "pv2": {"capacity_mw": 1584, "lat": 24.45, "lon": 54.6}}
CAP = {k: v["capacity_mw"] for k, v in SITES.items()}
DE = {"pv_sentinel_abs_mw": 5000, "night_neg_floor_mw": -10, "swap_date": "auto"}
REPORT = [f"# Solcast → Reuniwatt transfer analysis · run {datetime.utcnow():%Y-%m-%d %H:%M} UTC", f"settings: {W}"]


def as_text(df, title, max_rows=300):
    pdf = df if isinstance(df, pd.DataFrame) else df.limit(max_rows).toPandas()
    return f"## {title}\n" + (pdf.head(max_rows).round(3).to_string(index=False) if len(pdf) else "(no rows)") + "\n"


def show(df, title, max_rows=300):
    display(df)
    REPORT.append(as_text(df, title, max_rows))


def to15(df, ts, cols, by=("site",)):
    return (df.withColumn("period_end", ceil_to(ts, 15)).groupBy(*by, "period_end")
            .agg(*[F.avg(c).alias(c) for c in cols], F.count("*").alias("_n")))

## Load every source at 15-min (UTC, period end)
- **Target and sensors:** Bronze telemetry (Gulf → UTC; Mar 2026 column swap handled).
- **Solcast:** both Bronze tables (`pv1_history_solcast`, `pv2_history_solcast`).
- **Reuniwatt:** the pipeline table `uc2_int_reuniwatt_5min` (day-ahead runs, power in MW). Each 15-min period uses the average of runs issued **12–24 h** before it, a realistic day-ahead input.

In [ ]:
PV = spark.table("ewec_dev_powerops.bronze.power_contango_minute_pv").withColumn("ts_gulf", F.col("DateTime").cast("timestamp"))
SWAP = _detect_swap_date(PV, DE)
swap = F.lit(SWAP)
small = F.when(F.to_date("ts_gulf") < swap, _clean_mw("PV2_MW", DE)).otherwise(_clean_mw("PV1_MW", DE))
large = F.when(F.to_date("ts_gulf") < swap, _clean_mw("PV1_MW", DE)).otherwise(_clean_mw("PV2_MW", DE))
tel = reduce(DataFrame.unionByName, [
    PV.select(F.lit("pv1").alias("site"), gulf_to_utc("ts_gulf").alias("ts"), small.alias("actual_mw"),
              F.col("PV1_GHI").alias("m_ghi"), F.col("PV1_Temp").alias("m_temp")),
    PV.select(F.lit("pv2").alias("site"), gulf_to_utc("ts_gulf").alias("ts"), large.alias("actual_mw"),
              F.col("PV2_GHI").alias("m_ghi"), F.col("PV2_Outdoor_Temp").alias("m_temp")),
])
tel15 = to15(tel, "ts", ["actual_mw", "m_ghi", "m_temp"]).withColumnRenamed("_n", "n_min")
tel15 = tel15.withColumn("m_ghi", F.when(F.col("m_ghi") >= -20, F.greatest(F.col("m_ghi"), F.lit(0.0)))) \
             .withColumn("m_temp", F.when((F.col("m_temp") > 0) & (F.col("m_temp") < 60), F.col("m_temp")))

SC_VARS = {"ghi": "ghi", "dni": "dni", "dhi": "dhi", "gti": "gti", "air_temp": "temp", "cloud_opacity": "cloud"}
sc = {}
for t in ["pv1", "pv2"]:
    raw = spark.table(f"ewec_dev_powerops.bronze.{t}_history_solcast")
    have = {k: v for k, v in SC_VARS.items() if k in raw.columns}
    x = raw.select(F.col("period_end").cast("timestamp").alias("ts"), *[F.col(k).cast("double").alias(v) for k, v in have.items()])
    sc[t] = to15(x.withColumn("site", F.lit(t)), "ts", list(have.values())).drop("_n")

RU_VARS = {"ghi": "ghi", "dni": "dni", "dhi": "dhi", "gti_poa": "gti", "temp_c": "temp", "cloud_opacity": "cloud", "power_mw": "power"}
ru5 = spark.table(f"{S}.uc2_int_reuniwatt_5min").where("horizon = 'dayahead' AND lead_min > 720 AND lead_min <= 1440")
ru15 = to15(ru5.select("site", "period_end", *[F.col(k).alias(v) for k, v in RU_VARS.items()]), "period_end", list(RU_VARS.values())).drop("_n")
print("Bronze MW column swap:", SWAP)

# Part A · EDA (before any comparison or model)
**A1 · Coverage and null % by month** (daytime only: computed clear-sky GHI > 50). A month with rows but ~100% null is the trap we fell into before.

In [ ]:
def coverage(df, cols, label):
    d = add_solar(df, SITES, "period_end", 15).where("clearsky_ghi > 50")
    return (d.groupBy(F.lit(label).alias("source"), "site", F.date_trunc("month", "period_end").alias("month"))
            .agg(F.count("*").alias("periods"), *[F.round(100 * F.avg(F.col(c).isNotNull().cast("int")), 1).alias(f"{c}_filled_pct") for c in cols]))

cov = reduce(DataFrame.unionByName, [
    coverage(tel15, ["actual_mw", "m_ghi", "m_temp"], "telemetry").withColumnRenamed("actual_mw_filled_pct", "a_pct").withColumnRenamed("m_ghi_filled_pct", "b_pct").withColumnRenamed("m_temp_filled_pct", "c_pct"),
    *[coverage(sc[t], ["ghi", "temp", "cloud"], f"solcast_{t}_table").withColumnRenamed("ghi_filled_pct", "a_pct").withColumnRenamed("temp_filled_pct", "b_pct").withColumnRenamed("cloud_filled_pct", "c_pct") for t in sc],
    coverage(ru15, ["ghi", "temp", "power"], "reuniwatt_da").withColumnRenamed("ghi_filled_pct", "a_pct").withColumnRenamed("temp_filled_pct", "b_pct").withColumnRenamed("power_filled_pct", "c_pct"),
])
REPORT.append("A1 columns: telemetry a=actual_mw b=sensor GHI c=sensor temp | solcast a=ghi b=temp c=cloud | reuniwatt a=ghi b=temp c=power")
show(cov.orderBy("source", "site", "month"), "A1 · daytime coverage (% filled) by month", 500)

**A2 · Distribution of GHI by month** (daytime percentiles), per source. Look for empty months, unit changes, odd jumps, and differences in level between sources.

In [ ]:
def dist(df, col, label):
    d = add_solar(df, SITES, "period_end", 15).where("clearsky_ghi > 50")
    return (d.groupBy(F.lit(label).alias("source"), "site", F.date_trunc("month", "period_end").alias("month"))
            .agg(F.count(col).alias("n"), *[F.round(F.percentile_approx(col, q), 1).alias(f"p{int(q*100)}") for q in (0.05, 0.25, 0.5, 0.75, 0.95)],
                 F.round(F.max(col), 1).alias("max")))

show(reduce(DataFrame.unionByName, [dist(tel15, "m_ghi", "sensor"), *[dist(sc[t], "ghi", f"solcast_{t}_table") for t in sc],
                                    dist(ru15, "ghi", "reuniwatt_da")]).orderBy("source", "site", "month"),
     "A2 · daytime GHI distribution by month (W/m²)", 500)

**A3 · `satellite_weather_sts` (new table, profiled generically).** Finds it if the name is blank, then shows columns, date range, monthly coverage and null % per numeric column. The question to answer here: **is it a forecast (issue time / lead columns) or observed values?**

In [ ]:
sat_name = W["satellite_table"]
if not sat_name:
    hits = spark.sql("SELECT table_catalog, table_schema, table_name FROM system.information_schema.tables WHERE table_name = 'satellite_weather_sts'").collect()
    print("Found:", [f"{h[0]}.{h[1]}.{h[2]}" for h in hits])
    sat_name = f"{hits[0][0]}.{hits[0][1]}.{hits[0][2]}" if hits else ""
SAT = None
if sat_name:
    SAT = spark.table(sat_name)
    REPORT.append(f"## A3 · {sat_name}\ncolumns: " + ", ".join(f"{f.name} ({f.dataType.simpleString()})" for f in SAT.schema.fields))
    print(REPORT[-1])
    ts_cols = [f.name for f in SAT.schema.fields if f.dataType.typeName() in ("timestamp", "timestamp_ntz", "date")]
    num_cols = [f.name for f in SAT.schema.fields if f.dataType.typeName() in ("double", "float", "integer", "long", "decimal", "short")]
    print("timestamp-like:", ts_cols, "| numeric:", num_cols)
    if ts_cols:
        tcol = ts_cols[0]
        show(SAT.groupBy(F.date_trunc("month", F.col(tcol)).alias("month")).agg(
            F.count("*").alias("rows"), *[F.round(100 * F.avg(F.col(c).isNotNull().cast("int")), 1).alias(f"{c}_filled_pct") for c in num_cols[:12]]
        ).orderBy("month"), f"A3 · {sat_name}: rows and % filled by month (by {tcol})", 200)
        show(SAT.select(*[F.min(c).alias(f"min_{c}") for c in ts_cols] + [F.max(c).alias(f"max_{c}") for c in ts_cols]), f"A3 · {sat_name}: time ranges")
    display(SAT.limit(20))
else:
    REPORT.append("## A3 · satellite_weather_sts not found (set the widget to its full name)")
    print(REPORT[-1])

✅ **Read Part A:**
- **Solcast:** values until Nov 2025.
- **Reuniwatt:** values from ~Jul 2026.
- **Sensors and target:** all periods.
- **A2:** similar daytime GHI percentiles across sources means the sources share a scale. Different percentiles mean they need calibration.
- **A3:** the satellite table's months and columns show whether it fills the gap, and whether it's a forecast or observed values.

# Part B · Which Solcast table is which plant?
Correlation of each Solcast table's GHI with each plant's **own GHI sensor** (2024–2025). The best match is that plant's table. Earlier evidence suggested Solcast follows Phase 1's naming (its "pv1" table = the large plant).

In [ ]:
pairs = []
for t, d in sc.items():
    j = tel15.join(d.drop("site"), "period_end").where(F.to_date("period_end").between(F.lit(W["solcast_train_start"]), F.lit(W["solcast_train_end"])))
    pairs.append(j.groupBy("site").agg(F.round(F.corr("ghi", "m_ghi"), 4).alias("corr_with_sensor_ghi"),
                                        F.round(F.corr("ghi", "actual_mw"), 4).alias("corr_with_actual")).withColumn("solcast_table", F.lit(t)))
mapping_df = reduce(DataFrame.unionByName, pairs).orderBy("site", F.desc("corr_with_sensor_ghi"))
show(mapping_df, "B · Solcast table vs plant")
best = {r["site"]: r["solcast_table"] for r in mapping_df.withColumn("rk", F.row_number().over(Window.partitionBy("site").orderBy(F.desc("corr_with_sensor_ghi")))).where("rk = 1").collect()}
if len(set(best.values())) < 2:
    best = {"pv1": "pv1", "pv2": "pv2"}
    REPORT.append("B · mapping ambiguous → using same labels")
REPORT.append(f"B · Solcast table used per plant: {best}")
print(REPORT[-1])
sc_site = reduce(DataFrame.unionByName, [sc[t].withColumn("site", F.lit(s)) for s, t in best.items()])

# Part C · Same scale? Each source vs the plant sensors
The sensors cover every period, so they're the common reference even though Solcast and Reuniwatt don't overlap. Daytime only.
- **C1:** GHI and temperature agreement per source (Solcast 2024–25, Reuniwatt Jul 2026 onwards): correlation, bias, error % of mean.
- **C2:** distribution of the **clear-sky index** (GHI ÷ clear-sky) per source vs sensor. For a model trained on Solcast to work on Reuniwatt, both should look alike.

In [ ]:
base = add_solar(tel15, SITES, "period_end", 15).where("clearsky_ghi > 50")
sc_j = base.join(sc_site, ["site", "period_end"])
ru_j = base.join(ru15, ["site", "period_end"])


def agree(df, f, m, label, var):
    d = df.where(F.col(f).isNotNull() & F.col(m).isNotNull())
    return d.groupBy("site").agg(F.count("*").alias("n"), F.round(F.corr(f, m), 4).alias("corr"),
                                 F.round(F.avg(F.col(f) - F.col(m)), 2).alias("bias"),
                                 F.round(100 * F.avg(F.abs(F.col(f) - F.col(m))) / F.avg(F.abs(F.col(m))), 2).alias("err_pct_of_mean"),
                                 F.min("period_end").alias("from"), F.max("period_end").alias("to")).withColumn("source", F.lit(label)).withColumn("variable", F.lit(var))

c1 = reduce(DataFrame.unionByName, [agree(sc_j, "ghi", "m_ghi", "solcast", "ghi"), agree(ru_j, "ghi", "m_ghi", "reuniwatt_da_12_24h", "ghi"),
                                    agree(sc_j, "temp", "m_temp", "solcast", "temp"), agree(ru_j, "temp", "m_temp", "reuniwatt_da_12_24h", "temp")])
show(c1.orderBy("variable", "site", "source"), "C1 · agreement with plant sensors (daytime)")

def csi_dist(df, col, label):
    d = df.where(F.col(col).isNotNull()).withColumn("csi", F.least(F.col(col) / F.col("clearsky_ghi"), F.lit(1.5)))
    return d.groupBy("site").agg(F.count("*").alias("n"), *[F.round(F.percentile_approx("csi", q), 3).alias(f"p{int(q*100)}") for q in (0.05, 0.25, 0.5, 0.75, 0.95)],
                                 F.round(F.avg("csi"), 3).alias("mean")).withColumn("source", F.lit(label))

c2 = reduce(DataFrame.unionByName, [csi_dist(sc_j, "ghi", "solcast (2024–25)"), csi_dist(sc_j, "m_ghi", "sensor (2024–25)"),
                                    csi_dist(ru_j, "ghi", "reuniwatt (Jul 2026+)"), csi_dist(ru_j, "m_ghi", "sensor (Jul 2026+)")])
show(c2.orderBy("site", "source"), "C2 · clear-sky index distribution (daytime)")

✅ **Read Part C:**
- **Similar bias and error vs the sensors** for Solcast and Reuniwatt → same scale, so transfer is plausible.
- **In C2,** compare each source with the sensor *in its own period*. The gap between "Solcast vs sensor" and "Reuniwatt vs sensor" shows how differently the two sources behave.
- **A much smaller error for Solcast than for Reuniwatt** would mean Solcast is closer to *observed* weather, so a Solcast-trained model would be over-confident with Reuniwatt.

# Part D · Transfer test
Same features for every source (no GTI, because panel set-ups differ between sources; no cloud, because the units differ):
**GHI, DNI, DHI, temperature, clear-sky index**, plus computed **sun elevation, cos zenith, clear-sky GHI, hour and day of year**.

One LightGBM per plant (a **feasibility check**, not the production per-slot design), daytime rows, label = actual ÷ capacity.

| Model | Trained on | Tested on (same test window) |
|---|---|---|
| `S_raw` | Solcast, `solcast_train_start` → `solcast_train_end` | Reuniwatt features, as they are |
| `S_cal` | Solcast (same) | Reuniwatt features **calibrated to Solcast's scale** via the sensors (linear map per variable) |
| `R` | Reuniwatt, Jul 2026 → `reuni_train_end` | Reuniwatt features |
| `reuniwatt_power` | — | Reuniwatt's own power forecast (baseline) |
| `S_on_solcast` | Solcast up to Aug 2025 | Solcast Sep–Nov 2025 (how good Solcast is in its own world) |

In [ ]:
FEATS_SRC = ["ghi", "dni", "dhi", "temp"]
pi = 3.141592653589793

def frame(src_df, label):
    d = base.join(src_df.select("site", "period_end", *FEATS_SRC, *(["power"] if "power" in src_df.columns else [])), ["site", "period_end"])
    g = F.timestamp_seconds(F.unix_timestamp("period_end") + 4 * 3600)
    hr = F.hour(g) + F.minute(g) / 60
    d = (d.withColumn("csi", F.least(F.col("ghi") / F.col("clearsky_ghi"), F.lit(1.5)))
          .withColumn("hour_sin", F.sin(2 * pi * hr / 24)).withColumn("hour_cos", F.cos(2 * pi * hr / 24))
          .withColumn("doy_sin", F.sin(2 * pi * F.dayofyear(g) / 365.25)).withColumn("doy_cos", F.cos(2 * pi * F.dayofyear(g) / 365.25))
          .withColumn("gulf_hour", hr).withColumn("source", F.lit(label)))
    return d

feat_cols = FEATS_SRC + ["csi", "sun_elevation", "cos_zenith", "clearsky_ghi", "hour_sin", "hour_cos", "doy_sin", "doy_cos"]
keep = ["site", "period_end", "actual_mw", "m_ghi", "m_temp", "n_min", "gulf_hour", "is_day", "source"] + feat_cols
both = frame(sc_site, "solcast").select(*keep, F.lit(None).cast("double").alias("power")).unionByName(frame(ru15, "reuniwatt").select(*keep, "power"))
both.write.mode("overwrite").option("overwriteSchema", "true").saveAsTable(f"{S}.uc2_analysis_transfer_15min")
pdf = spark.table(f"{S}.uc2_analysis_transfer_15min").toPandas()
pdf["period_end"] = pd.to_datetime(pdf["period_end"])
pdf["day"] = pdf["period_end"].dt.date
pdf["cap"] = pdf["site"].map(CAP)
pdf["y"] = pdf["actual_mw"] / pdf["cap"]
win = pdf["is_day"].astype(bool) if W["daytime_mode"] == "sun" else (pdf["gulf_hour"] > 6) & (pdf["gulf_hour"] <= 18)
pdf = pdf[win & pdf["y"].notna() & (pdf["n_min"] >= 8)]
print(pdf.groupby(["source", "site"]).agg(rows=("y", "size"), first=("day", "min"), last=("day", "max")))

**Calibration for `S_cal`:** for each variable and plant, fit sensor ≈ a·source + b separately for Solcast (its period) and Reuniwatt (its training period). Then map Reuniwatt values onto Solcast's scale: x_solcast = (a_r·x + b_r − b_s) / a_s.

In [ ]:
def d_(s):
    return date.fromisoformat(s)

test_end = d_(W["test_end"]) if W["test_end"] else pdf.loc[pdf.source == "reuniwatt", "day"].max()
sc_tr = pdf[(pdf.source == "solcast") & (pdf.day >= d_(W["solcast_train_start"])) & (pdf.day <= d_(W["solcast_train_end"]))]
ru_all = pdf[pdf.source == "reuniwatt"]
ru_tr = ru_all[ru_all.day <= d_(W["reuni_train_end"])]
ru_te = ru_all[(ru_all.day >= d_(W["test_start"])) & (ru_all.day <= test_end)]
REPORT.append(f"D · rows: Solcast train {len(sc_tr)} | Reuniwatt train {len(ru_tr)} | test {len(ru_te)} ({W['test_start']} → {test_end})")
print(REPORT[-1])

CAL_VARS = {"ghi": "m_ghi", "dni": "m_ghi", "dhi": "m_ghi", "temp": "m_temp"}  # dni/dhi have no sensor: scaled with the GHI map
cal = {}
for site in CAP:
    for v, ref in CAL_VARS.items():
        fits = {}
        for name, d in [("s", sc_tr), ("r", ru_tr)]:
            dd = d[(d.site == site)][["ghi" if v in ("dni", "dhi") else v, ref]].dropna()
            fits[name] = np.polyfit(dd.iloc[:, 0], dd.iloc[:, 1], 1) if len(dd) > 50 else np.array([1.0, 0.0])
        cal[(site, v)] = fits
REPORT.append("D · calibration (sensor = a·source + b): " + "; ".join(f"{s}/{v}: solcast a={f['s'][0]:.3f} b={f['s'][1]:.1f}, reuniwatt a={f['r'][0]:.3f} b={f['r'][1]:.1f}" for (s, v), f in cal.items()))
print(REPORT[-1])


def calibrate(d):
    d = d.copy()
    for site in CAP:
        m = d.site == site
        for v in ("ghi", "dni", "dhi", "temp"):
            (a_s, b_s), (a_r, b_r) = cal[(site, v)]["s"], cal[(site, v)]["r"]
            d.loc[m, v] = (a_r * d.loc[m, v] + b_r - b_s) / a_s
        d.loc[m, "csi"] = np.minimum(d.loc[m, "ghi"] / d.loc[m, "clearsky_ghi"], 1.5)
    return d

In [ ]:
def fit(d):
    import lightgbm as lgb
    m = lgb.LGBMRegressor(n_estimators=500, learning_rate=0.03, num_leaves=31, min_child_samples=30, subsample=0.8, subsample_freq=1,
                          colsample_bytree=0.8, verbose=-1)
    m.fit(d[feat_cols].astype("float32"), d["y"])
    return m


rows = []
preds = []
for site in CAP:
    te = ru_te[ru_te.site == site].copy()
    if te.empty:
        continue
    m_s = fit(sc_tr[sc_tr.site == site])
    variants = {"S_raw": m_s.predict(te[feat_cols].astype("float32")),
                "S_cal": m_s.predict(calibrate(te)[feat_cols].astype("float32"))}
    r_tr = ru_tr[ru_tr.site == site]
    if len(r_tr) > 500:
        variants["R"] = fit(r_tr).predict(te[feat_cols].astype("float32"))
    for name, p in variants.items():
        te[f"pred_{name}"] = np.clip(p, 0, 1.05) * CAP[site]
    te["pred_reuniwatt_power"] = te["power"]
    for name in list(variants) + ["reuniwatt_power"]:
        rows.append({"site": site, "model": name, **score(te.assign(cap_mw=CAP[site]), f"pred_{name}", "actual_mw", "cap_mw", 150)})
    preds.append(te)
    # Solcast in its own world: train before Sep 2025, test Sep 1 – Nov 18 2025
    s_all = pdf[(pdf.source == "solcast") & (pdf.site == site)]
    s_tr, s_te = s_all[s_all.day < date(2025, 9, 1)], s_all[(s_all.day >= date(2025, 9, 1)) & (s_all.day <= date(2025, 11, 18))].copy()
    if len(s_te):
        s_te["pred"] = np.clip(fit(s_tr).predict(s_te[feat_cols].astype("float32")), 0, 1.05) * CAP[site]
        rows.append({"site": site, "model": "S_on_solcast (Sep–Nov 2025)", **score(s_te.assign(cap_mw=CAP[site]), "pred", "actual_mw", "cap_mw", 150)})

res = pd.DataFrame(rows).sort_values(["site", "mape_pct"])
show(res, "D · transfer test (MAPE where actual > 150 MW; nMAE/RMSE/bias % of capacity)")

**D2 · By Gulf hour** (test window): where does the Solcast-trained model lose against the Reuniwatt-trained model and the baseline?

In [ ]:
if preds:
    allp = pd.concat(preds)
    allp["hour"] = allp["gulf_hour"].apply(lambda h: int(np.ceil(h)) - 1)
    hr_rows = []
    for (site, h), g in allp.groupby(["site", "hour"]):
        for name in [c[5:] for c in allp.columns if c.startswith("pred_")]:
            hr_rows.append({"site": site, "gulf_hour": h, "model": name, **score(g.assign(cap_mw=CAP[site]), f"pred_{name}", "actual_mw", "cap_mw", 150)})
    show(pd.DataFrame(hr_rows).pivot_table(index=["site", "gulf_hour"], columns="model", values="mape_pct").round(2).reset_index(),
         "D2 · MAPE by Gulf hour and model", 100)

✅ **How to decide**
| Outcome | Meaning |
|---|---|
| `S_raw` or `S_cal` ≈ `R`, and ≤ `reuniwatt_power` | **Train on Solcast, infer on Reuniwatt works.** Use the calibrated version if it's clearly better |
| `S_cal` good but `S_raw` poor | It works **only with calibration**. Keep the calibration step in production |
| Both clearly worse than `R` | Transfer doesn't work well enough. Wait for Reuniwatt history, or train on Reuniwatt only |
| `S_on_solcast` much better than any Reuniwatt result | Solcast history behaves like near-observed weather. A Solcast-trained model will be over-confident with real forecasts |

Caveat: `R` trains on only ~2 months, so it's a weak reference. It will improve as Reuniwatt history grows.

In [ ]:
text = "\n".join(REPORT)
os.makedirs("results", exist_ok=True)
with open("results/solcast_reuniwatt_transfer.txt", "w") as fh:
    fh.write(text)
print(f"Saved {os.path.abspath('results/solcast_reuniwatt_transfer.txt')} — commit & push it.\n")
print(text)